# <b> CRBL MF v25</b>

<b> authors: Roberta Maria Lorenzi </b>

<b> version 1) February 8th, 2025 </b>

contacts: robertamaria.lorenzi01@universitadipavia.it

## <b> Test Configurations </b>


## What is new?
Update version of Cerebellar mean-field (CRBL-MF v25) based on a mouse-awake configurations


Synaptic and connectivity is set following same rationale of PLOS2023 (CRBL-MF v23):
- 'CRBL_CONFIG_20PARALLEL_wN_PLOS23' for details on K and Q tuning.
- alpha PC and MLI set at 5 in prediction as for CRBL-MF Fv23

Differences:
- Construction: SNN is the canonical awake. NO PARAMETERS TUNING FOR MEAN FIELD DERIVATION. IN MFv23, the network has mossy fibers-goc connectivity reduced to have Kmf-goc = 35 AND 20% parallel fibers convergence.
-  Prediction: alpha GoC in prediction is 1.5 (3 in fitting), to prevent GrC to be at 0.



## Mean field model simulations

Testing the configuration PLOS23 with different values of alpha

In [1]:
#import the libraries
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
sys.path.append('../')
import os

In [2]:
#from google.colab import drive
#drive.mount('/content/gdrive')

In [3]:
### PUT HERE THE PATH WHERE YOU SAVED THE DATA
# gdrive.MyDrive is the gdrive home
#from gdrive.MyDrive.MF_awake_pred_zmin.MF_prediction.load_config_TF import *
#from gdrive.MyDrive.MF_awake_pred_zmin.MF_prediction.master_equation_CRBL_MF import *
#from gdrive.MyDrive.MF_awake_pred_zmin.MF_prediction.theoretical_tools import *

In [4]:
from load_config_TF import *
from master_equation_CRBL_MF import *
from theoretical_tools import *

In [5]:
root_path = os.getcwd()+'/' #folder where P coefficients were stored
NTWK = 'CRBL_CONFIG_20PARALLEL_wN_PLOS23_Kgocgrc_red'

In [6]:
print(root_path)

/home/bcc/projects/BSB4_demo/cerebellum_zmin_Ie1.2/


In [7]:
Ngrc = 29916
Ngoc = 71
Nmossy = 2340
Nmli = 302+150
Npc = 69

In [8]:
dt = 1e-4
sim_len = 0.5

In [9]:
T = 3.5e-3
w = 0. #adaptation not included at the moment

In [10]:
#root_path = '/home/bcc/projects/BSB4_demo/cerebellum_zmin_Ie1.2/' #folder where P coefficients were stored

NRN1, NRN2, NRN3, NRN4 = 'GrC', 'GoC', 'MLI', 'PC'
#NTWK = 'CRBL_CONFIG_20PARALLEL_wN_PLOS23'

FILE_GrC = root_path  + '20250131_161943_GrC_CRBL_CONFIG_20PARALLEL_wN_PLOS23_tsim5_alpha1.2_fit.npy'
FILE_GoC = root_path + '20250203_165809_GoC_CRBL_CONFIG_20PARALLEL_wN_PLOS23_tsim5_alpha3.0_fit.npy'
FILE_MLI = root_path + '20250206_160547_MLI_CRBL_CONFIG_20PARALLEL_wN_PLOS23_tsim5_alpha1.8_fit.npy'

FILE_PC = root_path + '20250206_160227_PC_CRBL_CONFIG_20PARALLEL_wN_PLOS23_tsim5_alpha2.6_fit.npy' #Zmin
alphas = np.loadtxt('alpha_2025_02_11.txt')

print(alphas)

TFgrc = load_transfer_functions(NRN1, NTWK, FILE_GrC, alpha=alphas[0]) #2 ## SE RIMETTO 2, sfascia tutto.
TFgoc = load_transfer_functions_goc(NRN2, NTWK, FILE_GoC, alpha=alphas[1]) # Provo a dimezzare alpha GoC visto che in rete SNN NON ho 20 parallel
TFmli = load_transfer_functions(NRN3, NTWK, FILE_MLI, alpha=alphas[2]) # Se non metto alfa 5 anche in MLI non ho "picco pausa" 
TFpc = load_transfer_functions(NRN4, NTWK, FILE_PC, alpha=alphas[3]) 


[2.  1.5 5.  5. ]
synaptic network_scaffold parameters in SI units [V, F, s]
..................... Loading TFs
Neuron and Network:  GrC CRBL_CONFIG_20PARALLEL_wN_PLOS23_Kgocgrc_red
Loaded P file:  /home/bcc/projects/BSB4_demo/cerebellum_zmin_Ie1.2/20250131_161943_GrC_CRBL_CONFIG_20PARALLEL_wN_PLOS23_tsim5_alpha1.2_fit.npy


[-0.41245767 -0.00640348  0.02120219  0.21117751  0.0936675 ]
synaptic network_scaffold parameters in SI units [V, F, s]
..................... Loading TFs
Neuron and Network:  GoC CRBL_CONFIG_20PARALLEL_wN_PLOS23_Kgocgrc_red
Loaded P file:  /home/bcc/projects/BSB4_demo/cerebellum_zmin_Ie1.2/20250203_165809_GoC_CRBL_CONFIG_20PARALLEL_wN_PLOS23_tsim5_alpha3.0_fit.npy


synaptic network_scaffold parameters in SI units [V, F, s]
..................... Loading TFs
Neuron and Network:  MLI CRBL_CONFIG_20PARALLEL_wN_PLOS23_Kgocgrc_red
Loaded P file:  /home/bcc/projects/BSB4_demo/cerebellum_zmin_Ie1.2/20250206_160547_MLI_CRBL_CONFIG_20PARALLEL_wN_PLOS23_tsim5_alpha1.8_fit.np

In [11]:
t = np.arange(0, sim_len, dt)

In [12]:
def plot_MF_activity(t, X):
    fig = go.Figure()

    fig = make_subplots(rows=5, cols=1)

    fig.add_trace(go.Scatter(x=t, y=X[:,0],
                                 mode='lines',
                                 name='GrC',
                                 line=dict(color = "red")),
                      row=4, col=1
                      )

    fig.add_trace(go.Scatter(x=t, y=X[:,1],
                                 mode='lines',
                                 name='GoC',
                                 line=dict(color = "blue")),
                      row=3, col=1
                      )
    fig.add_trace(go.Scatter(x=t, y=X[:,9],
                                 mode='lines',
                                 name='MLI',
                                 line=dict(color = "orange")),
                      row=2, col=1
                      )

    fig.add_trace(go.Scatter(x=t, y=X[:,10],
                                 mode='lines',
                                 name='PC',
                                 line=dict(color = "green")),
                      row=1, col=1
                      )

    fig.add_trace(go.Scatter(x=t, y=X[:,8],
                                 mode='lines',
                                 name='mossy fibers',
                                 line=dict(color = "black")),
                      row=5, col=1
                      )


    fig.update_xaxes(title_text="time [s]", row=5, col=1)


    # Update yaxis properties
    fig.update_yaxes(title_text="Activity [Hz]", row=3, col=1)
    """
    fig.update_yaxes(title_text="GrC[Hz]", row=4, col=1)
    fig.update_yaxes(title_text="GoC[Hz]", row=3, col=1)
    fig.update_yaxes(title_text="MLI[Hz]", row=2, col=1)
    fig.update_yaxes(title_text="PC[Hz]", row=1, col=1)
    fig.update_yaxes(title_text="mf[Hz]", row=5, col=1)
    """
    axis_style = dict(
      showline=True,      
      linecolor='black',  
      linewidth=1,        
      showgrid=False,     
      zeroline=False,     
      mirror=False        #true to have the box
    )


    # Update title
    fig.update_layout(
                      title_text='CRBL Zmin Mean Field Prediction',
                      height=600, width=400,
                      plot_bgcolor = 'rgba(0,0,0,0)', #remove bckg
                      xaxis=axis_style, xaxis2=axis_style, xaxis3=axis_style, xaxis4=axis_style, xaxis5=axis_style,
                      yaxis=axis_style, yaxis2=axis_style, yaxis3=axis_style, yaxis4=axis_style, yaxis5=axis_style,
                      font=dict(family="Arial, sans-serif", size=12, color="black")
    )

    

    for i, idx in enumerate([10, 9, 1, 0, 8]):  # Indici delle colonne in X
      y_min, y_max = np.min(X[:, idx]), np.max(X[:, idx])
      tick_values = np.linspace(y_min, y_max, 3)
      fig.update_yaxes(tickmode="array", tickvals=tick_values, tickformat=".0f", row=i+1, col=1)

    fig.show()
    

In [13]:
def rect_input(time, t_start, t_end, minval, freq, noise_freq):

    """
    time = time vector of simulation
    t_start = start of the step INDEX
    t_end = end of the step INDEX
    minval = baseline value (deviation from 0)
    freq = peak value
    noise_freq = random noise frequencies
    """

    y = np.ones(len(time)) * freq + np.random.rand(len(time)) * noise_freq
    y[:t_start] = y[:t_start]*0+np.random.rand(t_start)*noise_freq
    y[t_end:] = y[t_end:]*0+np.random.rand(len(time) - t_end)*noise_freq
    y = y + minval

    return y

**Cortical-like**

In [14]:
f_backnoise = np.random.rand(len(t))*4 #background noise set around 4 Hz
#f_backnoise = np.ones_like(t)*0
amplitude =7.5
freq1 = 15
freq2 = 30
freq3 = 1
f_sin1 = amplitude * np.sin(2 * np.pi * freq1 * t) + amplitude
f_sin2 = amplitude * np.sin(2*np.pi*freq2*t)+ amplitude
f_sin3 = amplitude * np.sin(2 * np.pi * freq3 * t) + amplitude

fmulti_sin = f_sin1 + f_sin2 + f_sin3
fmossy_cort = fmulti_sin*0 + f_backnoise

In [15]:
# X = vector of activity with Vp = population p mean activity, Csp = covariance between population s and p

# X = [Vgrc, Vgoc, Cgrcgrc, Cgrcgoc, Cgrcm, Cmgoc, Cgocgoc, Cmm, Vm, Vmli, Vpc, Cmlimli, Cmlipc, Cgrcpc, Cgrcmli,
#     Cpcpc, Cmligoc, Cmlimossy, Cpcgoc, Cpcmossy]


CI_vec = [0.5, 5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, fmossy_cort[0], 15, 38, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]

X_cort = find_fixed_point_mossy(TFgrc, TFgoc, TFmli, TFpc, CI_vec, t, w, fmossy_cort,
                           Ngrc, Ngoc, Nmossy, Nmli, Npc, T, verbose=False)

In [16]:
plot_MF_activity(t[1000:], X_cort[1000:,:])

In [17]:
print(fmossy_cort)

[0.99749075 1.08166666 3.50154716 ... 0.98780001 1.27558543 2.3641968 ]


<b> Sensory-like stimulus </b>

In [18]:
f_tone = rect_input(time=t, t_start=1250, t_end=3750, minval=0, freq=16, noise_freq=0)
fmossy_sens = fmossy_cort + f_tone
print(np.max(fmossy_sens))

19.99952616169491


In [19]:
# X = vector of activity with Vp = population p mean activity, Csp = covariance between population s and p

# X = [Vgrc, Vgoc, Cgrcgrc, Cgrcgoc, Cgrcm, Cmgoc, Cgocgoc, Cmm, Vm, Vmli, Vpc, Cmlimli, Cmlipc, Cgrcpc, Cgrcmli,
#     Cpcpc, Cmligoc, Cmlimossy, Cpcgoc, Cpcmossy]


CI_vec = [0.5, 5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, fmossy_sens[0], 15, 38, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]

X_sens = find_fixed_point_mossy(TFgrc, TFgoc, TFmli, TFpc, CI_vec, t, w, fmossy_sens,
                           Ngrc, Ngoc, Nmossy, Nmli, Npc, T, verbose=False)

In [20]:
plot_MF_activity(t[1000:], X_sens[1000:,:])

In [21]:
f_tone2 = rect_input(time=t, t_start=1250, t_end=3750, minval=0, freq=10, noise_freq=0)
fmossy_sens2 = f_backnoise + f_tone2
print(np.max(fmossy_sens2))

# X = vector of activity with Vp = population p mean activity, Csp = covariance between population s and p

# X = [Vgrc, Vgoc, Cgrcgrc, Cgrcgoc, Cgrcm, Cmgoc, Cgocgoc, Cmm, Vm, Vmli, Vpc, Cmlimli, Cmlipc, Cgrcpc, Cgrcmli,
#     Cpcpc, Cmligoc, Cmlimossy, Cpcgoc, Cpcmossy]


CI_vec = [0.5, 5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, fmossy_sens[0], 15, 38, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]

X_tone = find_fixed_point_mossy(TFgrc, TFgoc, TFmli, TFpc, CI_vec, t, w, fmossy_sens2,
                           Ngrc, Ngoc, Nmossy, Nmli, Npc, T, verbose=False)

plot_MF_activity(t[1000:], X_tone[1000:,:])

13.999526161694911


In [22]:
f_tone_train = rect_input(time=t, t_start=1250, t_end=1300, minval=0, freq=200, noise_freq=0)
fmossy_sens_train = f_backnoise + f_tone_train
print(np.max(fmossy_sens_train))

# X = vector of activity with Vp = population p mean activity, Csp = covariance between population s and p

# X = [Vgrc, Vgoc, Cgrcgrc, Cgrcgoc, Cgrcm, Cmgoc, Cgocgoc, Cmm, Vm, Vmli, Vpc, Cmlimli, Cmlipc, Cgrcpc, Cgrcmli,
#     Cpcpc, Cmligoc, Cmlimossy, Cpcgoc, Cpcmossy]


CI_vec = [0.5, 5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, fmossy_sens[0], 15, 38, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]

X_tone_train = find_fixed_point_mossy(TFgrc, TFgoc, TFmli, TFpc, CI_vec, t, w, fmossy_sens_train,
                           Ngrc, Ngoc, Nmossy, Nmli, Npc, T, verbose=False)

plot_MF_activity(t[1000:], X_tone_train[1000:,:])

203.88242218970015


## References:

- Cerebellar MF model: Lorenzi et al. 2023 (https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1011434)
- E-GLIF: Geminiani et al. 2018, 2019 (https://doi.org/10.3389/fninf.2018.00088 and https://doi.org/10.3389/fncom.2019.00035)
- BSB: De Schepper et al. 2022 (https://doi.org/10.1038/s42003-022-04213-y)
- Mathematics: Boustani and Destexhe 2009 (https://doi.org/10.1162/neco.2009.02-08-710)
- TF formalism: Zerlaut et al. 2018 (https://doi.org/10.1007/s10827-017-0668-2)
- TF Fitting procedure: Zerlaut et al. 2016  (https://doi.org/10.1113/JP272317)
- MF with adaptation: Di Volo et al. 2019 (https://doi.org/10.1162/neco_a_01173)
- MF review for different neuron model: Carlu et al. 2020 (https://doi.org/10.1152/jn.00399.2019)